In [ ]:
import json 
import numpy as np 

with open('/home/david/Desktop/yuna/HPA/evaluation/logits/pretrained/llava-v1.6-vicuna-7b-hf/vqa_1k.jsonl') as f:
    data_list = [json.loads(l) for l in f]
example = data_list[0] 

print(example['question'], "Ouptut:", example['output'], 'Answer:', example['multiple_choice_answer']) 
token_probs = np.exp([t['logprob'] for t in example['logprobs_data']['content']]) 
confidence = token_probs.mean()

print("Token probabilities:", token_probs)
print("Confidence:", confidence)

Question: What is the woman in gray doing? Answer the question using a single word or phrase. 
Answer: Ouptut: Taking picture Answer: taking picture
Token probabilities: [0.79452022 0.97728923 0.62029676 0.99965965]
Confidence: 0.8479414649955388


In [ ]:
import os 

def get_example_confidence(model='llava-v1.6-vicuna-7b-hf', dataset='vqa_1k_control'): 
    filepath = f'/home/david/Desktop/yuna/HPA/evaluation/logits/pretrained/{model}/{dataset}.jsonl' 
    if not os.path.exists(filepath): 
        print(filepath, 'does not exist')
        return 
    with open(filepath) as f:
        data_list = [json.loads(l) for l in f]
    
    if len(data_list) == 0: 
        os.remove(filepath)
        print(f'deleted', filepath )
        return 

    example = data_list[0]  
    print(f'MODEL: {model} | DATASET: {dataset}') 

    if 'control' in dataset: 
        for q in example['generated_logits'].keys():  
            # print(example['question'], "Output:", example['output'], 'Answer:', example['multiple_choice_answer']) 
            logits = example['generated_logits'][q]['content']
            token_probs = np.exp([t['logprob'] for t in logits])
            confidence = token_probs.mean()
            print(f"{q}: {example[q]} \nAnswer: {data_list[0]['answers'][q]} conf: {confidence:.2f}")
    else: 
        token_probs = np.exp([t['logprob'] for t in example['logprobs_data']['content']]) 
        confidence = token_probs.mean()
        print(
            example['question'],
            "Output:", example['output'],
            f"conf: {confidence:.2f}"
            f"\nGT Answer: {example['multiple_choice_answer']}"
        )
        
    return data_list 

In [61]:
models = ['llava-v1.6-vicuna-7b-hf'] # "Qwen3-VL-4B-Instruct"]  
datasets = ['vqa_1k', 'vqa_1k_inst_blind', 'vqa_1k_control', 'vqa_1k_control_inst_blind']

for model in models: 
    for ds in datasets: 
        get_example_confidence(model, ds)

MODEL: llava-v1.6-vicuna-7b-hf | DATASET: vqa_1k
Question: What is the woman in gray doing? Answer the question using a single word or phrase. 
Answer: Output: Taking picture conf: 0.85
GT Answer: taking picture
/home/david/Desktop/yuna/HPA/evaluation/logits/pretrained/llava-v1.6-vicuna-7b-hf/vqa_1k_inst_blind.jsonl does not exist
MODEL: llava-v1.6-vicuna-7b-hf | DATASET: vqa_1k_control
question: What number of clocks are on this tower? 
Answer: 1 conf: 0.98
deictic_removed: What number of clocks are on the tower? 
Answer: 1 conf: 0.98
object_removed: What number of objects are on this tower? 
Answer: 1 conf: 0.83
weaker_object: What number of devices are on this tower? 
Answer: 1 conf: 0.90
subject_ablated: What number of the objects are on this tower? 
Answer: 12 conf: 0.82
MODEL: llava-v1.6-vicuna-7b-hf | DATASET: vqa_1k_control_inst_blind


In [ ]:
df = analyze_confidences(results)
df.head()

,image_id,question_id,conf_question,pred_question,correct_question,acc_question,conf_deictic_removed,pred_deictic_removed,correct_deictic_removed,acc_deictic_removed,...,correct_object_removed,acc_object_removed,conf_weaker_object,pred_weaker_object,correct_weaker_object,acc_weaker_object,conf_subject_ablated,pred_subject_ablated,correct_subject_ablated,acc_subject_ablated
0,524577,524577006,0.999989,<|im_end|>,1,False,0.999986,<|im_end|>,1,False,...,1,False,0.976570,<|im_end|>,1,False,0.000000,2,12,False
1,524577,524577024,0.000000,building,on building,False,0.000000,building,on building,False,...,outside,False,0.733708,<|im_end|>,wall,False,0.665584,<|im_end|>,outside,False
2,524577,524577027,0.000000,abic,arabic,False,0.000000,abic,arabic,False,...,arabic,False,0.000000,abic,arabic,False,0.000000,abic,arabic,False
3,524799,524799000,0.360309,<|im_end|>,6,False,0.360309,<|im_end|>,6,False,...,young,False,0.393762,<|im_end|>,7,False,0.409214,<|im_end|>,young,False
4,524850,524850001,0.999949,<|im_end|>,2,False,0.999949,<|im_end|>,2,False,...,2,False,0.885704,<|im_end|>,1,False,0.657866,<|im_end|>,2,False
